In [1]:
import sys
from pathlib import Path
raiz = Path.cwd()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
from src.cuantizacion import (ajustar_cuantizador, transformar, reconstruir,
                              calcular_error, ESTRATEGIAS, REPRESENTANTES)
DIR_TABLAS = raiz / "resultados" / "actividad3" / "tablas"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

In [2]:
rng = np.random.default_rng(0)
conjuntos = {
    "sintetica_lognormal": rng.lognormal(mean=0.0, sigma=1.0, size=100_000),
    "ingresos_1M_muestra": pd.read_parquet(
        raiz / "data" / "dataset_1M.parquet", columns=["ingresos"]
    )["ingresos"].sample(100_000, random_state=0).to_numpy(),
}
BINS = [5, 10, 25, 50]

In [3]:
filas = []
for nombre, datos in conjuntos.items():
    for estrategia in ESTRATEGIAS:
        for k in BINS:
            m = ajustar_cuantizador(datos, estrategia, k, columna=nombre)
            cod = transformar(datos, m)
            for r in REPRESENTANTES:
                e = calcular_error(datos, reconstruir(cod, m, r))
                filas.append({
                    "datos": nombre, "estrategia": estrategia, "representante": r,
                    "bins_solicitados": k, "bins_reales": m.n_bins_real,
                    "mae": e["mae"], "rmse": e["rmse"], "error_max": e["error_max"],
                })

comp = pd.DataFrame(filas)
comp.to_csv(DIR_TABLAS / "comparacion_representantes.csv", index=False)
print(comp.shape)       # (48, 8)
comp.head(12)

(48, 8)


,datos,estrategia,representante,bins_solicitados,bins_reales,mae,rmse,error_max
0,sintetica_lognormal,uniforme,punto_medio,5,5,9.785821,9.933396,11.350640
1,sintetica_lognormal,uniforme,media,5,5,1.228590,1.961990,21.090693
2,sintetica_lognormal,uniforme,mediana,5,5,1.101994,2.060874,21.716768
3,sintetica_lognormal,uniforme,punto_medio,10,10,4.273490,4.450426,5.675320
4,sintetica_lognormal,uniforme,media,10,10,1.120580,1.622070,9.826296
5,sintetica_lognormal,uniforme,mediana,10,10,1.021600,1.711506,10.370650
6,sintetica_lognormal,uniforme,punto_medio,25,25,1.302277,1.438196,2.270128
7,sintetica_lognormal,uniforme,media,25,25,0.792804,1.002757,3.321079
8,sintetica_lognormal,uniforme,mediana,25,25,0.754418,1.048821,3.632659
9,sintetica_lognormal,uniforme,punto_medio,50,50,0.552876,0.632806,1.135064


In [4]:
comp.pivot_table(index=["datos", "estrategia", "bins_solicitados"],
                 columns="representante", values=["mae", "rmse"]).round(4)

mae             \
representante                                        media    mediana   
datos               estrategia bins_solicitados                         
ingresos_1M_muestra cuantiles  5                 3149.7810  3030.7032   
                               10                1825.3377  1760.6365   
                               25                 869.0255   840.7248   
                               50                 488.9852   475.1410   
                    uniforme   5                 6056.0889  6011.9209   
                               10                3433.5185  3410.5005   
                               25                1569.9265  1566.8122   
                               50                 804.6718   803.9117   
sintetica_lognormal cuantiles  5                    0.5110     0.4661   
                               10                   0.3304     0.3040   
                               25                   0.1799     0.1668   
                               50                   0.1117     0.1042   
                    uniforme   5                    1.2286     1.1020   
                               10                   1.1206     1.0216   
                               25                   0.7928     0.7544   
                               50                   0.4860     0.4772   

                                                                  rmse  \
representante                                   punto_medio      media   
datos               estrategia bins_solicitados                          
ingresos_1M_muestra cuantiles  5                 12291.6649  5975.7238   
                               10                 5938.0929  4340.6260   
                               25                 2255.3097  2885.1533   
                               50                 1080.7269  2133.0697   
                    uniforme   5                  6569.0181  7268.8988   
                               10                 4002.9005  4089.6542   
                               25                 1622.1275  1827.2175   
                               50                  812.2707   930.0119   
sintetica_lognormal cuantiles  5                    10.7712     1.5271   
                               10                    5.3041     1.2948   
                               25                    2.0683     1.0304   
                               50                    1.0097     0.8628   
                    uniforme   5                     9.7858     1.9620   
                               10                    4.2735     1.6221   
                               25                    1.3023     1.0028   
                               50                    0.5529     0.5804   

                                                                        
representante                                      mediana punto_medio  
datos               estrategia bins_solicitados                         
ingresos_1M_muestra cuantiles  5                 6231.7879  24610.9433  
                               10                4534.1726  16075.7957  
                               25                3011.4452   9043.5921  
                               50                2232.4439   5782.2951  
                    uniforme   5                 7334.2691   7894.0185  
                               10                4127.0489   4638.6381  
                               25                1831.6074   1873.1270  
                               50                 931.2075    936.6480  
sintetica_lognormal cuantiles  5                    1.5978     23.8762  
                               10                   1.3567     16.5360  
                               25                   1.0799     10.1010  
                               50                   0.9018      6.9109  
                    uniforme   5                    2.0609      9.9334  
                               10                   1.7115      4.4504  
                   

In [5]:
piv = comp.pivot_table(index=["datos", "estrategia", "bins_solicitados"],
                       columns="representante", values=["mae", "rmse"])
ok_rmse = (piv["rmse"]["media"] <= piv["rmse"]["punto_medio"] + 1e-9).all()
ok_mae = (piv["mae"]["mediana"] <= piv["mae"]["media"] + 1e-9).all()
print("RMSE(media) <= RMSE(punto_medio):", ok_rmse)
print("MAE(mediana) <= MAE(media):", ok_mae)

RMSE(media) <= RMSE(punto_medio): True
MAE(mediana) <= MAE(media): True
